## 0. Setup Ambiente (Locale / Colab)
Rileva automaticamente se stiamo eseguendo su Google Colab e prepara il filesystem ad alte prestazioni per evitare i colli di bottiglia di Google Drive.

In [11]:
import sys
import os
import shutil

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Ambiente Colab rilevato. Montaggio di Google Drive in corso...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Percorso base del progetto su Drive (modificare se la cartella ha un nome diverso)
    BASE_DIR = '/content/drive/MyDrive/Progetto Machine Learning'
    
    # Installa le dipendenze mancanti su Colab
    print("Installazione dipendenze (rich, pyyaml)...")
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'rich', 'pyyaml'])
    
    if os.path.exists(BASE_DIR):
        os.chdir(BASE_DIR)
        sys.path.append(BASE_DIR)
        print(f"Directory di lavoro impostata su: {BASE_DIR}")
    else:
        print(f"\nATTENZIONE: La cartella {BASE_DIR} non esiste su Drive!")
        print("Modifica la variabile BASE_DIR per puntare alla cartella in cui hai salvato il progetto.")
else:
    print("Ambiente locale rilevato. Nessuna operazione speciale richiesta.")


Ambiente locale rilevato. Nessuna operazione speciale richiesta.


#  Waste Classifier Trainer

## Sistema di addestramento configurabile per classificazione rifiuti

Questo notebook permette di configurare e eseguire training di modelli di deep learning per la classificazione di immagini di rifiuti.

### Modelli supportati:
- EfficientNet-B0, B2, B3
- MobileNetV3-Small

### Funzionalità:
- Dataset automatico con augmentation adattiva per classe
- K-Fold Cross Validation
- Scheduler LR automatico
- Salvataggio completo: pesi, grafici, log

In [12]:
# Importa librerie, widget e funzioni del modulo locale.

import sys
import os
from pathlib import Path
from datetime import datetime
import json
import yaml

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import models, transforms, datasets
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from tqdm.notebook import tqdm
from PIL import Image

sys.path.append('.')
from waste_classifier.trainer import (
    AdaptiveAugmentationDataset, ModelFactory, Trainer, ExperimentManager,
    extract_dataset, analyze_dataset, get_default_augmentation_strategies, advanced_stratified_split, analyze_dataset_with_rich, print_dataset_structure_with_rich
# Seleziona automaticamente GPU se disponibile, altrimenti CPU.
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.59 GB


In [13]:
# Caricamento dei default dal file config.yaml
try:
    with open('config.yaml', 'r') as f:
        default_config = yaml.safe_load(f)
    print("config.yaml caricato correttamente. I widget useranno questi valori come default.")
except Exception as e:
    print("Impossibile caricare config.yaml, verranno usati i default di base.")
    default_config = {}


config.yaml caricato correttamente. I widget useranno questi valori come default.


## 1 Configurazione Dataset

In [ ]:
# Crea i widget per scegliere dataset, ZIP, percentuali di split e seed.
dataset_path_widget = widgets.Text(
    value='/content/dataset_locale/waste_type_identification' if 'google.colab' in sys.modules else './dataset/waste_type_identification',
    description='Dataset:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

zip_path_widget = widgets.Text(
    value='./waste_type_identification.zip',
    description='ZIP:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

train_ratio_widget = widgets.FloatSlider(
    value=default_config.get('dataset', {}).get('train_ratio', 0.70), min=0.50, max=0.90, step=0.05,
    description='Train %:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

random_seed_widget = widgets.IntText(
    value=default_config.get('dataset', {}).get('random_seed', 42),
    description='Seed:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

extract_button = widgets.Button(
    description='Carica Dataset',
    button_style='primary',
    layout=widgets.Layout(width='auto')
)

# Estrae o carica il dataset, poi calcola classi e numero di immagini.
dataset_output = widgets.Output()

def on_extract_clicked(b):
    with dataset_output:
        clear_output()
        global DATASET_PATH, DATASET_STATS, CLASS_NAMES, N_CLASSES
        
        zip_path = zip_path_widget.value
        dataset_name = 'waste_type_identification'
        
        IN_COLAB = 'google.colab' in sys.modules
        if IN_COLAB:
            extract_to = '/content/dataset_locale'
            print("Utilizzo memoria locale Colab per l'estrazione ultraveloce...")
            local_zip = '/content/temp_dataset.zip'
            if not os.path.exists(local_zip):
                import shutil
                print("Copia dello ZIP da Drive a locale in corso (richiede qualche istante)...")
                shutil.copyfile(zip_path, local_zip)
            zip_path = local_zip
        else:
            extract_to = './dataset'
            
        DATASET_PATH = extract_dataset(zip_path, extract_to, dataset_name)
        DATASET_STATS = analyze_dataset(DATASET_PATH)
        CLASS_NAMES = DATASET_STATS['class_names']
        N_CLASSES = DATASET_STATS['num_classes']
        
        print(f"Dataset caricato con successo!")
        print_dataset_structure_with_rich(str(DATASET_PATH))
        global full_dataset, targets, indices, train_indices, test_indices, val_indices, stratify_labels
        full_dataset = datasets.ImageFolder(DATASET_PATH, transform=None)
        targets = np.array(full_dataset.targets)
        indices = list(range(len(full_dataset)))
        
        train_ratio = train_ratio_widget.value
        train_indices, test_indices, val_indices, stratify_labels = advanced_stratified_split(
            str(DATASET_PATH), full_dataset.samples, split_type='static',
            train_ratio=train_ratio, random_seed=random_seed_widget.value
        )
        
        print(f"Split avanzato completato:")
        analyze_dataset_with_rich(stratify_labels, train_indices, test_indices, val_indices)

extract_button.on_click(on_extract_clicked)

display(widgets.VBox([
    widgets.HTML('<h3>Configurazione Dataset</h3>'),
    zip_path_widget,
    dataset_path_widget,
    widgets.VBox([train_ratio_widget, random_seed_widget]),
    extract_button,
    dataset_output
]))

## 2 Selezione Modello e Configurazione Blocchi

### Come funziona la configurazione dei blocchi

Le reti EfficientNet sono composte da **8 blocchi** (indicizzati da 0 a 7):

```
Input → [Block 0] → [Block 1] → ... → [Block 5] → [Block 6] → [Block 7] → Classifier
         │                                                              │
         └────────────────── Feature Extraction ─────────────────────────┘
```

| Blocchi | Cosa apprendono |
|---------|----------------|
**0-2** | Feature di base (bordi, colori, texture semplici) |
**3-4** | Feature intermedie (forme, pattern) |
**5-7** | Feature avanzate (oggetti, parti specifiche) |

**Regola generale:**
- **Blocchi bassi (0-2)** → quasi sempre congiati (feature universali)
- **Blocchi alti (5-7)** → da sbloccare per adattare al dominio specifico

Seleziona quali blocchi **sbloccare** per il fine-tuning:

In [15]:
# Crea i controlli per scegliere modello, pesi pre-addestrati e dropout.
model_selector = widgets.SelectMultiple(
    options=['efficientnet_b0', 'efficientnet_b2', 'efficientnet_b3', 'mobilenet_v3_small'],
    value=['efficientnet_b0'],
    description='Modelli:',
    layout=widgets.Layout(width='auto', height='120px'),
    style={'description_width': 'initial'}
)

pretrained_checkbox = widgets.Checkbox(
    value=default_config.get('models', {}).get('efficientnet_b0', {}).get('pretrained', True),
    description='Usa pesi pre-addestrati',
    style={'description_width': 'initial'}
)

dropout_widget = widgets.FloatSlider(
    value=default_config.get('models', {}).get('efficientnet_b0', {}).get('dropout', 0.3), min=0.0, max=0.5, step=0.05,
    description='Dropout:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

effnet_options = [
    ('Blocco 1 (Stem)', 0),
    ('Blocco 2', 1),
    ('Blocco 3', 2),
    ('Blocco 4', 3),
    ('Blocco 5', 4),
    ('Blocco 6', 5),
    ('Blocco 7', 6),
    ('Blocco 8', 7),
    ('Blocco 9 (Conv Head)', 8)
]

unfreeze_efficientnet = widgets.SelectMultiple(
    options=effnet_options,
    value=[7, 8],
    description='Sblocca blocchi:',
    layout=widgets.Layout(width='auto', height='150px'),
    style={'description_width': 'initial'}
)

unfreeze_mobilenet = widgets.IntSlider(
    value=3, min=1, max=10, step=1,
    description='Ultimi N blocchi MobileNet:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

model_config_output = widgets.Output()
unfreeze_header = widgets.HTML('<h4>Configurazione Blocchi (Fine-Tuning)</h4>')
unfreeze_instruction = widgets.HTML('<small><i>(Tieni premuto <b>Ctrl</b> o <b>Cmd</b> cliccando per selezionare più blocchi singoli)</i></small>')

def update_model_config(change=None):
# Mostra un riepilogo della configurazione scelta prima del training.
    is_pretrained = pretrained_checkbox.value
    models_selected = list(model_selector.value)
    
    has_efficientnet = any('efficientnet' in m for m in models_selected)
    has_mobilenet = any('mobilenet' in m for m in models_selected)
    
    unfreeze_efficientnet.layout.display = 'flex' if (is_pretrained and has_efficientnet) else 'none'
    unfreeze_mobilenet.layout.display = 'flex' if (is_pretrained and has_mobilenet) else 'none'
    unfreeze_instruction.layout.display = 'block' if (is_pretrained and has_efficientnet) else 'none'
    unfreeze_header.layout.display = 'block' if (is_pretrained and (has_efficientnet or has_mobilenet)) else 'none'

    with model_config_output:
        clear_output()
        print(f"Modelli selezionati: {models_selected}")
        print(f"Pretrained: {is_pretrained}")
        print(f"Dropout: {dropout_widget.value}")
        
        for m in models_selected:
            if not is_pretrained:
                print(f"{m}:")
                print("RETE VUOTA: Tutti i blocchi sono sbloccati per un addestramento completo da zero.")
            elif 'efficientnet' in m:
                unfrozen_display = [i+1 for i in sorted(unfreeze_efficientnet.value)]
                frozen_display = [i+1 for i in range(9) if (i+1) not in unfrozen_display]
                print(f"{m}:")
                print(f"Blocchi CONGELATI: {frozen_display}")
                print(f"Blocchi SBLOCCATI: {unfrozen_display}")
            elif 'mobilenet' in m:
                print(f"{m}:")
                print(f"Ultimi {unfreeze_mobilenet.value} blocchi sbloccati")

model_selector.observe(update_model_config)
pretrained_checkbox.observe(update_model_config)
dropout_widget.observe(update_model_config)
unfreeze_efficientnet.observe(update_model_config)
unfreeze_mobilenet.observe(update_model_config)

# Initialize display state
update_model_config()

display(widgets.VBox([
    widgets.HTML('<h3>Selezione Modello</h3>'),
    model_selector,
    pretrained_checkbox,
    dropout_widget,
    unfreeze_header,
    widgets.VBox([unfreeze_instruction, unfreeze_efficientnet, unfreeze_mobilenet]),
    model_config_output
]))



## 3 Configurazione Augmentation

In [16]:
# Crea i controlli per configurare la data augmentation.
global_aug_p = widgets.FloatSlider(
    value=default_config.get('augmentation', {}).get('global_aug_p', 0.6), min=0.0, max=1.0, step=0.1,
    description='Prob. augmentation:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

use_class_specific = widgets.Checkbox(
    value=default_config.get('augmentation', {}).get('use_class_specific', True),
    description='Usa strategie per classe',
    style={'description_width': 'initial'}
)

rotation_range = widgets.IntSlider(
    value=default_config.get('augmentation', {}).get('default_strategy', {}).get('rotation_range', 15), min=0, max=90, step=5,
    description='Rotazione:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

zoom_range = widgets.FloatSlider(
    value=default_config.get('augmentation', {}).get('default_strategy', {}).get('zoom_range', 0.1), min=0.0, max=0.5, step=0.05,
    description='Zoom:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

horizontal_flip = widgets.Checkbox(
    value=default_config.get('augmentation', {}).get('default_strategy', {}).get('horizontal_flip', True),
    description='↔ Flip orizzontale',
    style={'description_width': 'initial'}
)

brightness_min = widgets.FloatSlider(
    value=default_config.get('augmentation', {}).get('default_strategy', {}).get('brightness_range', [0.8, 1.2])[0] if 'brightness_range' in default_config.get('augmentation', {}).get('default_strategy', {}) else 0.8, min=0.0, max=1.0, step=0.1,
    description='Lum. min:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

brightness_max = widgets.FloatSlider(
    value=default_config.get('augmentation', {}).get('default_strategy', {}).get('brightness_range', [0.8, 1.2])[1] if 'brightness_range' in default_config.get('augmentation', {}).get('default_strategy', {}) else 1.2, min=1.0, max=2.0, step=0.1,
    description='Lum. max:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

add_noise = widgets.Checkbox(
    value=default_config.get('augmentation', {}).get('default_strategy', {}).get('add_noise', False),
    description='Rumore gaussiano',
    style={'description_width': 'initial'}
)

aug_base_header = widgets.HTML('<h4>Parametri Base</h4>')
aug_preview = widgets.Output()

# Visualizza un riepilogo delle strategie di augmentation selezionate.
def update_aug_preview(change=None):
    is_disabled = (global_aug_p.value == 0.0)
    use_class_specific.disabled = is_disabled
    rotation_range.disabled = is_disabled
    zoom_range.disabled = is_disabled
    horizontal_flip.disabled = is_disabled
    brightness_min.disabled = is_disabled
    brightness_max.disabled = is_disabled
    add_noise.disabled = is_disabled
    
    if is_disabled:
        aug_base_header.value = '<h4>Parametri Disabilitati</h4>'
    elif use_class_specific.value:
        aug_base_header.value = '<h4>Parametri di Fallback <small><i>(applicati solo a: vetro, carta, vestiti)</i></small></h4>'
    else:
        aug_base_header.value = '<h4>Parametri Globali <small><i>(applicati a TUTTE le classi)</i></small></h4>'
        
    with aug_preview:
        clear_output()
        print("Strategie di augmentation:")
        print(f"Globale:")
        print(f"Probabilità: {global_aug_p.value*100:.0f}%")
        print(f"Rotazione: ±{rotation_range.value}°")
        print(f"Zoom: ±{zoom_range.value*100:.0f}%")
        print(f"Flip: {horizontal_flip.value}")
        print(f"Luminosità: [{brightness_min.value}, {brightness_max.value}]")
        print(f"Rumore: {add_noise.value}")
        
        if use_class_specific.value and not is_disabled:
            print(f"\nStrategie per classe: ATTIVE (Sovrascrivono i parametri sopra per le classi specifiche)")
            strategies = get_default_augmentation_strategies()
            for class_name, params in strategies.items():
                print(f" - {class_name}: rot={params['rotation_range']}°, zoom={params['zoom_range']}")
        else:
            print(f"\nStrategie per classe: DISATTIVE")

global_aug_p.observe(update_aug_preview)
use_class_specific.observe(update_aug_preview)
rotation_range.observe(update_aug_preview)
zoom_range.observe(update_aug_preview)
horizontal_flip.observe(update_aug_preview)
brightness_min.observe(update_aug_preview)
brightness_max.observe(update_aug_preview)
add_noise.observe(update_aug_preview)

update_aug_preview()

display(widgets.VBox([
    widgets.HTML('<h3>Configurazione Augmentation</h3>'),
    global_aug_p,
    use_class_specific,
    aug_base_header,
    rotation_range,
    zoom_range,
    horizontal_flip,
    widgets.VBox([brightness_min, brightness_max]),
    add_noise,
    aug_preview
]))


## 4 Configurazione Training

In [17]:
# Crea i controlli per batch size, epoche, learning rate e scheduler.
batch_size_widget = widgets.Dropdown(
    options=[16, 32, 64, 128],
    value=default_config.get('training', {}).get('batch_size', 32),
    description='Batch size:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

use_amp = widgets.Checkbox(
    value=default_config.get('training', {}).get('use_amp', True),
    description='Mixed Precision (AMP)',
    style={'description_width': 'initial'}
)

fe_epochs = widgets.IntSlider(
    value=default_config.get('training', {}).get('feature_extraction', {}).get('epochs', 10), min=1, max=50, step=1,
    description='Epoche FE:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

ft_epochs = widgets.IntSlider(
    value=default_config.get('training', {}).get('fine_tuning', {}).get('epochs', 30), min=1, max=100, step=1,
    description='Epoche FT:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

lr_fe_widget = widgets.FloatLogSlider(
    value=default_config.get('training', {}).get('feature_extraction', {}).get('learning_rate', 1e-3), base=10, min=-6, max=-1, step=0.1,
    description='LR Feature Extraction:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

lr_ft_widget = widgets.FloatLogSlider(
    value=default_config.get('training', {}).get('fine_tuning', {}).get('learning_rate', 1e-5), base=10, min=-7, max=-2, step=0.1,
    description='LR Fine Tuning:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

patience_fe_widget = widgets.IntSlider(
    value=default_config.get('training', {}).get('feature_extraction', {}).get('patience', 5), min=1, max=20, step=1,
    description='⏳ Patience FE:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

min_delta_fe_widget = widgets.FloatLogSlider(
    value=default_config.get('training', {}).get('feature_extraction', {}).get('min_delta', 1e-3), base=10, min=-5, max=-1, step=0.1,
    description='Min Delta FE:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

patience_ft_widget = widgets.IntSlider(
    value=default_config.get('training', {}).get('fine_tuning', {}).get('patience', 7), min=1, max=20, step=1,
    description='⏳ Patience FT:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

min_delta_ft_widget = widgets.FloatLogSlider(
    value=default_config.get('training', {}).get('fine_tuning', {}).get('min_delta', 5e-4), base=10, min=-5, max=-1, step=0.1,
    description='Min Delta FT:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

weight_decay_widget = widgets.FloatLogSlider(
    value=default_config.get('training', {}).get('weight_decay', 1e-4), base=10, min=-6, max=-1, step=0.1,
    description='Weight Decay:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

use_scheduler = widgets.Checkbox(
    value=default_config.get('training', {}).get('scheduler', {}).get('enabled', True),
# Configura il tipo di scheduler da usare durante il training.
    description='Usa Scheduler LR',
    style={'description_width': 'initial'}
)

scheduler_type = widgets.Dropdown(
    options=['ReduceLROnPlateau', 'CosineAnnealingLR', 'StepLR'],
    value=default_config.get('training', {}).get('scheduler', {}).get('type', 'ReduceLROnPlateau'),
description='Tipo scheduler:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

scheduler_factor = widgets.FloatSlider(
    value=default_config.get('training', {}).get('scheduler', {}).get('factor', 0.3), min=0.1, max=0.9, step=0.1,
    description='Factor riduzione:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

use_kfold = widgets.Checkbox(
    value=default_config.get('kfold', {}).get('enabled', True),
    description='Usa K-Fold',
# Configura la validazione K-Fold, se abilitata.
    style={'description_width': 'initial'}
)

kfold_mode = widgets.Dropdown(
    options=['Valutativo + Modello Singolo', 'Ensemble di Modelli'],
    value='Valutativo + Modello Singolo',
    description='Modalità KF:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

n_splits = widgets.IntSlider(
    value=default_config.get('kfold', {}).get('n_splits', 3), min=2, max=10, step=1,
    description='N folds:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

def update_training_config(change=None):
    show_scheduler = use_scheduler.value
    scheduler_type.layout.display = 'flex' if show_scheduler else 'none'
    scheduler_factor.layout.display = 'flex' if show_scheduler else 'none'
    
    show_kfold = use_kfold.value
    n_splits.layout.display = 'flex' if show_kfold else 'none'
    kfold_mode.layout.display = 'flex' if show_kfold else 'none'

use_scheduler.observe(update_training_config)
use_kfold.observe(update_training_config)

# Inizializza lo stato visivo
update_training_config()

display(widgets.VBox([
    widgets.HTML('<h3>Configurazione Training</h3>'),
    widgets.HTML('<h4>Base</h4>'),
    widgets.VBox([batch_size_widget, use_amp]),
    widgets.VBox([fe_epochs, ft_epochs]),
    lr_fe_widget,
    lr_ft_widget,
    widgets.VBox([widgets.HBox([patience_fe_widget, min_delta_fe_widget]), widgets.HBox([patience_ft_widget, min_delta_ft_widget])]),
    weight_decay_widget,
    widgets.HTML('<h4>Scheduler</h4>'),
    use_scheduler,
    scheduler_type,
    scheduler_factor,
    widgets.HTML('<h4>K-Fold Cross Validation</h4>'),
    use_kfold,
    n_splits,
    kfold_mode
]))

## 5 Nome Esperimento

In [18]:
# Crea i controlli per nome esperimento e cartella di output.
experiment_name = widgets.Text(
    value='',
    description='Nome:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'},
placeholder='Opzionale - es: test_b0_augmented'
)

output_base_dir = widgets.Text(
    value='./experiments',
    description='Cartella output:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

estim_time = widgets.Output()

def estimate_time(change=None):
    with estim_time:
        clear_output()
        models_count = len(model_selector.value)
        total_epochs = fe_epochs.value + ft_epochs.value
# Aggiorna una stima indicativa del tempo in base ai parametri correnti.
        if use_kfold.value:
            total_epochs *= n_splits.value
        if use_scheduler.value:
            total_epochs = int(total_epochs * 0.7)  # Stima con early stopping
        
        sec_per_epoch = 30
        total_sec = total_epochs * sec_per_epoch * models_count
        total_min = total_sec / 60
        
        print(f"⏱ Tempo stimato: ~{total_min:.0f} minuti ({total_min/60:.1f} ore)")
        print(f"Modelli: {models_count}")
        print(f"Epoche totali (con FE+FT{'+KFOLD' if use_kfold.value else ''}): {total_epochs}")

model_selector.observe(estimate_time)
fe_epochs.observe(estimate_time)
ft_epochs.observe(estimate_time)
use_kfold.observe(estimate_time)
n_splits.observe(estimate_time)
use_scheduler.observe(estimate_time)

display(widgets.VBox([
    widgets.HTML('<h3>Esperimento</h3>'),
    experiment_name,
    output_base_dir,
    estim_time
]))

## 6 Avvio Training

In [19]:
# Avvia il training usando la configurazione selezionata nei widget precedenti.
train_button = widgets.Button(
    description='Avvia Training',
    button_style='success',
    layout=widgets.Layout(width='200px', height='50px')
)

training_output = widgets.Output(layout={'border': '1px solid gray'})

@train_button.on_click
def on_train_clicked(b):
    train_button.disabled = True
    stop_button.disabled = False
    try:
        with training_output:
            clear_output()
            global trainer
            
            if 'DATASET_PATH' not in globals():
                print("Devi prima caricare il dataset!")
                return
            
            models_selected = list(model_selector.value)
            if not models_selected:
                print("Seleziona almeno un modello!")
                return
            
            print("=" * 60)
            print("AVVIO TRAINING")
            print("=" * 60)
            
            exp_manager = ExperimentManager(output_base_dir.value)
            
            for model_name in models_selected:
                print(f"\n{'='*60}")
                print(f"Modello: {model_name}")
                print(f"{'='*60}")
                
                exp_dir = exp_manager.create_experiment_dir(
                    model_name, 
                    experiment_name.value if experiment_name.value else None
                )
                print(f"Esperimento: {exp_dir}")
                
                config = {
                    'model': {
                        'name': model_name,
                        'pretrained': pretrained_checkbox.value,
                        'dropout': dropout_widget.value,
                    },
                    'training': {
                        'batch_size': batch_size_widget.value,
                        'use_amp': use_amp.value,
                        'fe_epochs': fe_epochs.value,
                        'ft_epochs': ft_epochs.value,
                        'lr_fe': lr_fe_widget.value,
                        'lr_ft': lr_ft_widget.value,
                        'patience_fe': patience_fe_widget.value,
                        'min_delta_fe': min_delta_fe_widget.value,
                        'patience_ft': patience_ft_widget.value,
                        'min_delta_ft': min_delta_ft_widget.value,
                        'weight_decay': weight_decay_widget.value,
                        'label_smoothing': default_config.get('training', {}).get('label_smoothing', 0.0),
                    },
                    'augmentation': {
                        'global_aug_p': global_aug_p.value,
                        'use_class_specific': use_class_specific.value,
                    },
                    'scheduler': {
                        'enabled': use_scheduler.value,
                        'type': scheduler_type.value,
                        'factor': scheduler_factor.value,
                    },
                    'kfold': {
                        'enabled': use_kfold.value,
                        'n_splits': n_splits.value,
                    },
                    'dataset': {
                        'path': str(DATASET_PATH),
                        'train_ratio': train_ratio_widget.value,
                        'random_seed': random_seed_widget.value,
                    }
                }
                exp_manager.save_config(config, exp_dir)
                
                if 'b0' in model_name:
                    input_size = 224
                elif 'b2' in model_name:
                    input_size = 288
                elif 'b3' in model_name:
                    input_size = 300
                else:
                    input_size = 224
                
                if use_class_specific.value:
                    aug_strategies = get_default_augmentation_strategies()
                else:
                    default_params = {
                        'rotation_range': rotation_range.value,
                        'rotation_p': 0.5,
                        'horizontal_flip': horizontal_flip.value,
                        'horizontal_flip_p': 0.5,
                        'zoom_range': zoom_range.value,
                        'zoom_p': 0.5,
                        'brightness_range': [brightness_min.value, brightness_max.value],
                        'brightness_p': 0.5,
                        'add_noise': add_noise.value,
                        'add_noise_p': 0.5,
                    }
                    aug_strategies = {name: default_params.copy() for name in CLASS_NAMES}
                
                trainer = Trainer(config, exp_dir)
                
                if use_kfold.value:
                    fold_scores = []
                    trainval_indices = np.array(train_indices + val_indices)
                    trainval_targets = targets[trainval_indices]
                    
                    skf = StratifiedKFold(
                        n_splits=n_splits.value, 
                        shuffle=True, 
                        random_state=random_seed_widget.value
                    )
                    
                    for fold, (tr_pos, te_pos) in enumerate(skf.split(trainval_indices, trainval_targets), 1):
                        print(f"\n--- Fold {fold}/{n_splits.value} ---")
                        
                        fold_train_idx = trainval_indices[tr_pos]
                        fold_test_idx = trainval_indices[te_pos]
                        
                        fold_train_ds = AdaptiveAugmentationDataset(
                            Subset(full_dataset, fold_train_idx),
                            {i: n.lower() for i, n in enumerate(CLASS_NAMES)},
                            aug_strategies,
                            resize=input_size + 32,
                            crop=input_size,
                            is_train=True,
                            global_aug_p=global_aug_p.value
                        )
                        fold_test_ds = AdaptiveAugmentationDataset(
                            Subset(full_dataset, fold_test_idx),
                            {i: n.lower() for i, n in enumerate(CLASS_NAMES)},
                            aug_strategies,
                            resize=input_size + 32,
                            crop=input_size,
                            is_train=False
                        )
                        
                        fold_train_loader = DataLoader(
                            fold_train_ds, batch_size=batch_size_widget.value,
                            shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True
                        )
                        fold_test_loader = DataLoader(
                            fold_test_ds, batch_size=batch_size_widget.value,
                            shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True
                        )
                        
                        model = ModelFactory.create_model(
                            model_name, N_CLASSES, device,
                            pretrained=pretrained_checkbox.value,
                            dropout=dropout_widget.value
                        )
                        
                        fold_weights = trainer.compute_class_weights(fold_train_idx, targets, N_CLASSES)
                        fold_criterion = nn.CrossEntropyLoss(weight=fold_weights)
                        
                        if 'mobilenet' in model_name:
                            fe_params = model.classifier[3].parameters()
                        else:
                            fe_params = model.classifier[1].parameters()
                        opt_fe = optim.AdamW(fe_params, lr=lr_fe_widget.value, weight_decay=weight_decay_widget.value)
                        
                        hist_fe = trainer.train_model(
                            model, opt_fe, fold_train_loader, fold_test_loader, fold_criterion,
                            epochs=fe_epochs.value, patience=patience_fe_widget.value, min_delta=min_delta_fe_widget.value,
                            save_path=str(exp_dir / f"models/fold{fold}_fe.pth"),
                            label_smoothing=config['training'].get('label_smoothing', 0.0), use_amp=use_amp.value, phase_name=f"fold{fold}_feature_extraction"
                        )
                        
                        if 'efficientnet' in model_name:
                            ModelFactory.unfreeze_for_finetuning(
                                model, model_name, blocks=unfreeze_efficientnet.value
                            )
                        elif 'mobilenet' in model_name:
                            ModelFactory.unfreeze_for_finetuning(
                                model, model_name, n_last_blocks=unfreeze_mobilenet.value
                            )
                        
                        opt_ft = optim.AdamW(
                            filter(lambda p: p.requires_grad, model.parameters()),
                            lr=lr_ft_widget.value, weight_decay=weight_decay_widget.value
                        )
                        
                        scheduler = None
                        if use_scheduler.value:
                            if scheduler_type.value == 'ReduceLROnPlateau':
                                scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt_ft, mode='max', factor=scheduler_factor.value, patience=2)
                            elif scheduler_type.value == 'CosineAnnealingLR':
                                scheduler = optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=ft_epochs.value)
                            elif scheduler_type.value == 'StepLR':
                                scheduler = optim.lr_scheduler.StepLR(opt_ft, step_size=10, gamma=scheduler_factor.value)
                        
                        hist_ft = trainer.train_model(
                            model, opt_ft, fold_train_loader, fold_test_loader, fold_criterion,
                            epochs=ft_epochs.value, patience=patience_ft_widget.value, min_delta=min_delta_ft_widget.value,
                            save_path=str(exp_dir / f"models/fold{fold}_ft.pth"),
                            initial_best=max(hist_fe['val_bal_acc']), scheduler=scheduler,
                            label_smoothing=config['training'].get('label_smoothing', 0.0), use_amp=use_amp.value, phase_name=f"fold{fold}_fine_tuning"
                        )
                        
                        best_fold = max(max(hist_fe['val_bal_acc']), max(hist_ft['val_bal_acc']))
                        fold_scores.append(best_fold)
                        print(f"Fold {fold} best: {best_fold*100:.2f}%")
                        
                        exp_manager.save_history(hist_fe, exp_dir, filename=f"fold{fold}_fe_history.json")
                        exp_manager.save_history(hist_ft, exp_dir, filename=f"fold{fold}_ft_history.json")
                        exp_manager.save_training_curves(hist_ft, exp_dir, f"{model_name}_fold{fold}")
                    
                    print(f"\n{'='*40}")
                    print(f"K-Fold Val Estimation: {np.mean(fold_scores)*100:.2f}% (+/-{np.std(fold_scores)*100:.2f}%)")
                    print(f"{'='*40}")
                    
                    if kfold_mode.value == 'Valutativo + Modello Singolo':
                        print("\n=== ADDESTRAMENTO MODELLO FINALE SULL'INTERO TRAINING SET ===")
                        from sklearn.model_selection import train_test_split
                        final_tr_idx, final_val_idx = train_test_split(
                            trainval_indices, test_size=0.15,
                            stratify=targets[trainval_indices], random_state=random_seed_widget.value
                        )
                        
                        final_train_ds = AdaptiveAugmentationDataset(
                            Subset(full_dataset, final_tr_idx), {i: n.lower() for i, n in enumerate(CLASS_NAMES)},
                            aug_strategies, resize=input_size+32, crop=input_size, is_train=True, global_aug_p=global_aug_p.value
                        )
                        final_val_ds = AdaptiveAugmentationDataset(
                            Subset(full_dataset, final_val_idx), {i: n.lower() for i, n in enumerate(CLASS_NAMES)},
                            aug_strategies, resize=input_size+32, crop=input_size, is_train=False
                        )
                        
                        final_train_loader = DataLoader(
                            final_train_ds, batch_size=batch_size_widget.value, shuffle=True,
                            num_workers=4, pin_memory=True, persistent_workers=True
                        )
                        final_val_loader = DataLoader(
                            final_val_ds, batch_size=batch_size_widget.value, shuffle=False,
                            num_workers=4, pin_memory=True, persistent_workers=True
                        )
                        
                        final_weights = trainer.compute_class_weights(final_tr_idx, targets, N_CLASSES)
                        final_criterion = nn.CrossEntropyLoss(weight=final_weights)
                        
                        final_model = ModelFactory.create_model(model_name, N_CLASSES, device, pretrained=pretrained_checkbox.value, dropout=dropout_widget.value)
                        
                        if 'mobilenet' in model_name:
                            fe_params = final_model.classifier[3].parameters()
                        else:
                            fe_params = final_model.classifier[1].parameters()
                        opt_fe = optim.AdamW(fe_params, lr=lr_fe_widget.value, weight_decay=weight_decay_widget.value)
                        
                        hist_fe = trainer.train_model(
                            final_model, opt_fe, final_train_loader, final_val_loader, final_criterion,
                            epochs=fe_epochs.value, patience=patience_fe_widget.value, min_delta=min_delta_fe_widget.value,
                            save_path=str(exp_dir / "models/final_fe.pth"),
                            label_smoothing=config['training'].get('label_smoothing', 0.0), use_amp=use_amp.value, phase_name="final_feature_extraction"
                        )
                        
                        if 'efficientnet' in model_name:
                            ModelFactory.unfreeze_for_finetuning(final_model, model_name, blocks=unfreeze_efficientnet.value)
                        elif 'mobilenet' in model_name:
                            ModelFactory.unfreeze_for_finetuning(final_model, model_name, n_last_blocks=unfreeze_mobilenet.value)
                        
                        opt_ft = optim.AdamW(
                            filter(lambda p: p.requires_grad, final_model.parameters()),
                            lr=lr_ft_widget.value, weight_decay=weight_decay_widget.value
                        )
                        
                        scheduler = None
                        if use_scheduler.value:
                            if scheduler_type.value == 'ReduceLROnPlateau':
                                scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt_ft, mode='max', factor=scheduler_factor.value, patience=2)
                            elif scheduler_type.value == 'CosineAnnealingLR':
                                scheduler = optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=ft_epochs.value)
                            elif scheduler_type.value == 'StepLR':
                                scheduler = optim.lr_scheduler.StepLR(opt_ft, step_size=10, gamma=scheduler_factor.value)
                                
                        hist_ft = trainer.train_model(
                            final_model, opt_ft, final_train_loader, final_val_loader, final_criterion,
                            epochs=ft_epochs.value, patience=patience_ft_widget.value, min_delta=min_delta_ft_widget.value,
                            save_path=str(exp_dir / "models/final_ft.pth"),
                            initial_best=max(hist_fe['val_bal_acc']), scheduler=scheduler,
                            label_smoothing=config['training'].get('label_smoothing', 0.0), use_amp=use_amp.value, phase_name="final_fine_tuning"
                        )
                        
                        exp_manager.save_history(hist_fe, exp_dir, filename="final_fe_history.json")
                        exp_manager.save_history(hist_ft, exp_dir, filename="final_ft_history.json")
                        exp_manager.save_training_curves(hist_ft, exp_dir, f"{model_name}_final")
                        model = final_model
                    else:
                        print("\n=== MODALITA' ENSEMBLE SELEZIONATA: SALTO L'ADDESTRAMENTO DEL MODELLO SINGOLO ===")

                else:
                    train_ds = AdaptiveAugmentationDataset(
                        Subset(full_dataset, train_indices), {i: n.lower() for i, n in enumerate(CLASS_NAMES)},
                        aug_strategies, resize=input_size+32, crop=input_size, is_train=True, global_aug_p=global_aug_p.value
                    )
                    test_ds = AdaptiveAugmentationDataset(
                        Subset(full_dataset, test_indices), {i: n.lower() for i, n in enumerate(CLASS_NAMES)},
                        aug_strategies, resize=input_size+32, crop=input_size, is_train=False
                    )
                    
                    train_loader = DataLoader(train_ds, batch_size=batch_size_widget.value, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
                    test_loader = DataLoader(test_ds, batch_size=batch_size_widget.value, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)
                    
                    model = ModelFactory.create_model(model_name, N_CLASSES, device, pretrained=pretrained_checkbox.value, dropout=dropout_widget.value)
                    
                    train_weights = trainer.compute_class_weights(train_indices, targets, N_CLASSES)
                    criterion = nn.CrossEntropyLoss(weight=train_weights)
                    
                    if 'mobilenet' in model_name:
                        fe_params = model.classifier[3].parameters()
                    else:
                        fe_params = model.classifier[1].parameters()
                    opt_fe = optim.AdamW(fe_params, lr=lr_fe_widget.value, weight_decay=weight_decay_widget.value)
                    
                    hist_fe = trainer.train_model(
                        model, opt_fe, train_loader, test_loader, criterion,
                        epochs=fe_epochs.value, patience=patience_fe_widget.value, min_delta=min_delta_fe_widget.value,
                        save_path=str(exp_dir / "models/final_fe.pth"),
                        label_smoothing=config['training'].get('label_smoothing', 0.0), use_amp=use_amp.value, phase_name="feature_extraction"
                    )
                    
                    if 'efficientnet' in model_name:
                        ModelFactory.unfreeze_for_finetuning(model, model_name, blocks=unfreeze_efficientnet.value)
                    elif 'mobilenet' in model_name:
                        ModelFactory.unfreeze_for_finetuning(model, model_name, n_last_blocks=unfreeze_mobilenet.value)
                    
                    opt_ft = optim.AdamW(
                        filter(lambda p: p.requires_grad, model.parameters()),
                        lr=lr_ft_widget.value, weight_decay=weight_decay_widget.value
                    )
                    
                    scheduler = None
                    if use_scheduler.value:
                        if scheduler_type.value == 'ReduceLROnPlateau':
                            scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt_ft, mode='max', factor=scheduler_factor.value, patience=2)
                        elif scheduler_type.value == 'CosineAnnealingLR':
                            scheduler = optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=ft_epochs.value)
                        elif scheduler_type.value == 'StepLR':
                            scheduler = optim.lr_scheduler.StepLR(opt_ft, step_size=10, gamma=scheduler_factor.value)
                            
                    hist_ft = trainer.train_model(
                        model, opt_ft, train_loader, test_loader, criterion,
                        epochs=ft_epochs.value, patience=patience_ft_widget.value, min_delta=min_delta_ft_widget.value,
                        save_path=str(exp_dir / "models/final_ft.pth"),
                        initial_best=max(hist_fe['val_bal_acc']), scheduler=scheduler,
                        label_smoothing=config['training'].get('label_smoothing', 0.0), use_amp=use_amp.value, phase_name="fine_tuning"
                    )
                    
                    exp_manager.save_history(hist_fe, exp_dir, filename="fe_history.json")
                    exp_manager.save_history(hist_ft, exp_dir, filename="ft_history.json")
                    exp_manager.save_training_curves(hist_ft, exp_dir, model_name)
                
                # --- FASE FINALE CONDIVISA: VALUTAZIONE SUL VERO TEST SET ---
                print("\n=== VALUTAZIONE SUL TEST SET FINALE ===")
                final_test_ds = AdaptiveAugmentationDataset(
                    Subset(full_dataset, test_indices), {i: n.lower() for i, n in enumerate(CLASS_NAMES)},
                    aug_strategies, resize=input_size+32, crop=input_size, is_train=False
                )
                final_test_loader = DataLoader(
                    final_test_ds, batch_size=batch_size_widget.value, shuffle=False,
                    num_workers=4, pin_memory=True, persistent_workers=True
                )
                
                y_true, y_pred = [], []
                if use_kfold.value and kfold_mode.value == 'Ensemble di Modelli':
                    print(f"Esecuzione inferenza Ensemble con {n_splits.value} modelli...")
                    ensemble_models = []
                    for fold in range(1, n_splits.value + 1):
                        m = ModelFactory.create_model(model_name, N_CLASSES, device, pretrained=False)
                        m.load_state_dict(torch.load(str(exp_dir / f"models/fold{fold}_ft.pth"), map_location=device))
                        m.eval()
                        ensemble_models.append(m)
                    with torch.no_grad():
                        for imgs, labels in final_test_loader:
                            imgs = imgs.to(device)
                            probs = torch.zeros(imgs.size(0), N_CLASSES).to(device)
                            for m in ensemble_models:
                                probs += torch.softmax(m(imgs), dim=1)
                            probs /= len(ensemble_models)
                            y_true.extend(labels.numpy())
                            y_pred.extend(probs.argmax(1).cpu().numpy())
                    model = ensemble_models[0] # per benchmark inferenza
                else:
                    model.eval()
                    with torch.no_grad():
                        for imgs, labels in final_test_loader:
                            imgs = imgs.to(device)
                            outputs = model(imgs)
                            y_true.extend(labels.numpy())
                            y_pred.extend(outputs.argmax(1).cpu().numpy())
                y_true = np.array(y_true)
                y_pred = np.array(y_pred)
                
                exp_manager.save_confusion_matrix(y_true, y_pred, CLASS_NAMES, exp_dir, model_name)
                report = exp_manager.save_classification_report(y_true, y_pred, CLASS_NAMES, exp_dir)
                
                inference_resources = trainer.benchmark_inference(model, final_test_loader, phase_name="inference")
                exp_manager.save_resource_usage(trainer.resource_tracker.records, exp_dir)
                
                ba = balanced_accuracy_score(y_true, y_pred)
                print(f"\n{'='*40}")
                print(f"Final Balanced Accuracy (su Test Set invisibile): {ba*100:.2f}%")
                fps = inference_resources.get('samples_per_second')
                peak_mb = inference_resources.get('gpu_peak_allocated_mb')
                if fps is not None:
                    print(f"Inferenza: {fps:.2f} immagini/s")
                if peak_mb is not None:
                    print(f"Picco GPU inferenza: {peak_mb:.1f} MB")
                print(f"Risorse salvate in: {exp_dir / 'logs' / 'resource_usage.json'}")
                print(f"\n{report}")
        
            print(f"\n{'='*60}")
            print(f"TRAINING COMPLETATO")
            print(f"Risultati salvati in: {exp_dir.parent}")
            print(f"{'='*60}")

    finally:
        train_button.disabled = False
        stop_button.disabled = True

stop_button = widgets.Button(
    description='Interrompi',
    button_style='danger',
    disabled=True,
    layout=widgets.Layout(width='200px', height='50px')
)

def on_stop_clicked(b):
    if 'trainer' in globals():
        trainer.stop_requested = True
        with training_output:
            print("\nRichiesta di interruzione ricevuta. Il training si fermerà in modo sicuro al termine dell'epoca corrente...")
            
stop_button.on_click(on_stop_clicked)

display(widgets.VBox([
    widgets.HTML('<h3>Avvio Training</h3>'),
    widgets.HBox([train_button, stop_button]),
    training_output
]))

## 7 Visualizza Risultati

In [20]:
# Elenca gli esperimenti salvati e permette di visualizzarli singolarmente tramite un menu a tendina.
from IPython.display import display, Image

experiments_dir = Path(output_base_dir.value) if 'output_base_dir' in globals() else Path('./experiments')

exp_dropdown = widgets.Dropdown(
    options=[],
    description='Esperimento:',
    layout=widgets.Layout(width='600px'),
    style={'description_width': 'initial'}
)

refresh_button = widgets.Button(
    description='Aggiorna Lista',
    button_style='info'
)

results_output = widgets.Output()

def get_experiment_list():
    if not experiments_dir.exists():
        return []
    
    exp_dirs = sorted(
        [d for d in experiments_dir.iterdir() if d.is_dir()],
        key=lambda x: x.stat().st_mtime,
        reverse=True
    )
    return [(d.name, d) for d in exp_dirs]

def update_dropdown(b=None):
    exps = get_experiment_list()
    if exps:
        exp_dropdown.options = exps
    else:
        exp_dropdown.options = [("Nessun esperimento trovato", None)]

def show_experiment_details(change=None):
    with results_output:
        clear_output()
        exp_dir = exp_dropdown.value
        
        if not exp_dir:
            return
            
        print(f"\n{'='*60}")
        print(f"ESPERIMENTO: {exp_dir.name}")
        print(f"{'='*60}")
        
        config_path = exp_dir / 'config.yaml'
        if config_path.exists():
            with open(config_path) as f:
                config = yaml.safe_load(f)
            model_name = config.get('model', {}).get('name', 'unknown')
            print(f"Modello utilizzato: {model_name}\n")
            
            # 1. Balanced Accuracy
            history_files = list((exp_dir / 'logs').glob('*history*.json'))
            best_acc = 0
            for hf in history_files:
                with open(hf) as f:
                    hist = json.load(f)
                if 'val_bal_acc' in hist:
                    best_acc = max(best_acc, max(hist['val_bal_acc']))
            print(f"Best Balanced Accuracy: {best_acc*100:.2f}%")
            
            # 2. Risorse hardware
            resource_path = exp_dir / 'logs' / 'resource_usage.json'
            if resource_path.exists():
                with open(resource_path) as f:
                    resources = json.load(f)
                inference = next((r for r in reversed(resources) if 'inference' in r.get('phase', '')), None)
                if inference:
                    fps = inference.get('samples_per_second')
                    peak = inference.get('gpu_peak_allocated_mb')
                    if fps is not None:
                        print(f"Velocità di inferenza: {fps:.2f} immagini/s")
                    if peak is not None:
                        print(f"Picco memoria GPU: {peak:.1f} MB")
            
            print(f"{'-'*60}\n")
            
            # 3. Report di classificazione
            report_path = exp_dir / 'logs' / 'classification_report.txt'
            if report_path.exists():
                print("CLASSIFICATION REPORT")
                with open(report_path, 'r') as f:
                    print(f.read())
                print(f"{'-'*60}\n")
            
            # 4. Immagini e Matrici di confusione
            plot_dir = exp_dir / 'plots'
            if plot_dir.exists():
                png_files = list(plot_dir.glob('*.png'))
                if png_files:
                    print("GRAFICI E MATRICI DI CONFUSIONE")
                    for img_path in png_files:
                        print(f"\n> {img_path.name}")
                        display(Image(filename=str(img_path)))
        else:
            print("Nessun file config.yaml trovato in questa cartella.")

refresh_button.on_click(update_dropdown)
exp_dropdown.observe(show_experiment_details, names='value')

# Inizializzazione
update_dropdown()

display(widgets.VBox([
    widgets.HTML('<h3>Risultati Esperimenti</h3>'),
    widgets.HBox([exp_dropdown, refresh_button]),
    results_output
]))


## 8 Analisi degli Errori con Grad-CAM

Questo blocco permette di ispezionare visivamente gli errori commessi dal modello sul Test Set. Tramite la **Grad-CAM** possiamo generare delle *heatmaps* (mappe di calore) che ci mostrano esattamente su quali parti dell'immagine la rete neurale si è concentrata prima di effettuare la sua predizione (errata).

Seleziona l'esperimento nel menu a tendina della cella precedente, e poi clicca il bottone qui sotto per avviare l'analisi.

In [21]:
import os
import io
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision.transforms.functional import to_pil_image
import ipywidgets as widgets
from IPython.display import display, clear_output
import yaml
from torch.utils.data import Subset, DataLoader
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split

try:
    from torchcam.methods import GradCAM
    from torchcam.utils import overlay_mask
except ImportError:
    print("Installazione di torchcam in corso...")
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torchcam"])
    from torchcam.methods import GradCAM
    from torchcam.utils import overlay_mask

gradcam_output = widgets.Output(layout={'border': '1px solid gray'})

gradcam_button = widgets.Button(
    description='Esegui Grad-CAM su Errori',
    button_style='warning',
    layout=widgets.Layout(width='300px', height='40px')
)

def analizza_errori_gradcam(b):
    with gradcam_output:
        clear_output(wait=True)
        exp_dir = exp_dropdown.value
        
        if not exp_dir:
            print("Per favore, seleziona un esperimento dal menu a tendina nella sezione precedente.")
            return
            
        config_path = exp_dir / 'config.yaml'
        if not config_path.exists():
            print(f"Errore: file config.yaml non trovato in {exp_dir}")
            return
            
        with open(config_path) as f:
            config = yaml.safe_load(f)
            
        model_name = config.get('model', {}).get('name')
        dataset_path = config.get('dataset', {}).get('path')
        
        if not model_name or not dataset_path:
            print("Configurazione dell'esperimento incompleta.")
            return
            
        print(f"Avvio analisi Grad-CAM per il modello {model_name}...")
        
        # 1. Caricamento Dataset e Test Indices
        try:
            full_dataset = ImageFolder(dataset_path, transform=None)
            targets = np.array(full_dataset.targets)
            indices = list(range(len(full_dataset)))
            _, test_indices, _, _ = advanced_stratified_split(
                dataset_path,
                full_dataset.samples,
                split_type='static',
                train_ratio=config['dataset']['train_ratio'],
                random_seed=config['dataset']['random_seed']
            )
            print(f'Test set selezionato: {len(test_indices)} immagini')
            class_names = full_dataset.classes
        except Exception as e:
            print(f"Errore nel caricamento del dataset: {e}")
            return
            
        if 'b0' in model_name: input_size = 224
        elif 'b2' in model_name: input_size = 288
        elif 'b3' in model_name: input_size = 300
        else: input_size = 224
            
        base_transform = AdaptiveAugmentationDataset._build_base_transform(resize=input_size+32, crop=input_size)
        test_ds = ImageFolder(dataset_path, transform=base_transform)
        
        # 2. Caricamento del Modello Finale
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = ModelFactory.create_model(model_name, len(class_names), device, pretrained=False)
        
        model_path = exp_dir / "models" / "final_ft.pth"
        if not model_path.exists():
            model_path = exp_dir / "models" / "fold1_ft.pth"
            if not model_path.exists():
                print(f"Nessun modello addestrato trovato in {exp_dir / 'models'}")
                return
            print("Modello finale non trovato (K-Fold usato per l'ensemble). Utilizzo il modello Fold 1 per l'analisi Grad-CAM visiva.")
            
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()
        
        # 3. Ricerca degli Errori
        print("Ricerca delle immagini classificate erroneamente nel test set...")
        loader = DataLoader(Subset(test_ds, test_indices), batch_size=32, shuffle=False, num_workers=2)
        errati, veri, predetti = [], [], []
        pos = 0
        
        with torch.no_grad():
            for imgs, labels in loader:
                preds = model(imgs.to(device)).argmax(1).cpu().numpy()
                labels_np = labels.numpy()
                for k in range(len(labels_np)):
                    if preds[k] != labels_np[k]:
                        errati.append(int(test_indices[pos + k]))
                        veri.append(labels_np[k])
                        predetti.append(preds[k])
                pos += len(labels_np)
                
        print(f"Trovati {len(errati)} errori su {len(test_indices)} immagini di test.")
        
        if not errati:
            print("Ottimo! Non ci sono errori nel test set con questo modello.")
            return
            
        # 4. Grad-CAM su un sottoinsieme di errori
        num_immagini = min(5, len(errati))
        print(f"Generazione Grad-CAM per le prime {num_immagini} immagini errate...")
        
        if 'efficientnet' in model_name:
            target_layers = ['features.4', 'features.6', 'features.8']
        elif 'mobilenet' in model_name:
            target_layers = ['features.10', 'features.13']
        else:
            target_layers = [list(dict(model.named_modules()).keys())[-2]]
            
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(device)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(device)
        
        for i in range(num_immagini):
            idx_reale = errati[i]
            vero_idx = veri[i]
            pred_idx = predetti[i]
            
            img_tensor, _ = test_ds[idx_reale]
            img_tensor = img_tensor.to(device)
            input_tensor = img_tensor.unsqueeze(0).requires_grad_(True)
            
            img_denorm = img_tensor * std + mean
            img_denorm = torch.clamp(img_denorm, 0, 1)
            img_pil = to_pil_image(img_denorm.cpu())
            
            fig, axes = plt.subplots(1, len(target_layers) + 1, figsize=(4 + 4*len(target_layers), 4))
            
            axes[0].imshow(img_pil)
            axes[0].set_title(f"Vero: {class_names[vero_idx]}\n Predetto: {class_names[pred_idx]}", color='red')
            axes[0].axis('off')
            
            for j, layer_name in enumerate(target_layers):
                try:
                    cam_extractor = GradCAM(model, target_layer=layer_name)
                    out = model(input_tensor)
                    activation_map = cam_extractor(int(pred_idx), out)
                    
                    risultato = overlay_mask(img_pil, to_pil_image(activation_map[0].squeeze(0), mode='F'), alpha=0.5)
                    
                    axes[j+1].imshow(risultato)
                    axes[j+1].set_title(f"Layer: {layer_name}")
                    axes[j+1].axis('off')
                    
                    cam_extractor.remove_hooks()
                except Exception as e:
                    axes[j+1].text(0.5, 0.5, f"N/A", ha='center', va='center')
                    axes[j+1].axis('off')
                    
            plt.tight_layout()
            plt.show()

gradcam_button.on_click(analizza_errori_gradcam)

display(widgets.VBox([
    widgets.HTML('<h3>🔍 Analisi Errori (Grad-CAM)</h3>'),
    widgets.HTML('<p>Usa questo strumento per capire <strong>dove</strong> il modello ha guardato quando ha sbagliato a classificare un\'immagine. Verranno visualizzate fino a 5 immagini errate.</p>'),
    gradcam_button,
    gradcam_output
]))